In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel, Kernel
from numpy.typing import NDArray
import numpy as np
from typing import Sequence
import matplotlib.pyplot as plt 


In [ ]:
kernel = (
    ConstantKernel(constant_value=2.0, constant_value_bounds=(1e-3, 1e3))
    * RBF(length_scale=1, length_scale_bounds=(1e-2, 1e2))
)

def exp_kernel(A: NDArray, B: NDArray) -> NDArray:
    # RBF kernel: k(xa, xb) = exp(-0.5 * ||xa - xb||^2) (lengthscale=1)
    A = np.asarray(A)  # (n_a, n_features)
    B = np.asarray(B)  # (n_b, n_features)

    # squared euclidean distance matrix (n_a, n_b)
    sqdist = np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=-1)

    return np.exp(-0.5 * sqdist)


In [ ]:
n_dim = 3
n_per_dim = 5

x = np.linspace(-1, 1, n_per_dim)
g = np.meshgrid(*([x]*n_dim))

# Create the full grid and reshape to (n_samples, 3)
X = np.vstack([gi.ravel() for gi in g]).T  # (25, 3)

gram_A = kernel(X, X)
gram_B = 2.0*exp_kernel(X, X)


fig, axs = plt.subplots(1, 3)

axs[0].imshow(gram_A, cmap='viridis', aspect="auto")
axs[0].set_title("scipy SE kernel")
axs[1].imshow(gram_B, cmap='viridis', aspect="auto")
axs[1].set_title("Homemade SE kernel")
axs[2].imshow(gram_B- gram_A, cmap='viridis', aspect="auto")
axs[2].set_title("Difference")
plt.tight_layout()
plt.show()

### Hilbert Space Gaussian Process

In [ ]:
class SquaredExponentialKernel:
    def __init__(self, sigma_f: float = 1.0, len_scale: float = 1.0):
        self.sigma_f = sigma_f
        self.len_scale = len_scale

    def __call__(self, A: NDArray, B: NDArray) -> NDArray:
        A = np.asarray(A)  # (n_a, n_features)
        B = np.asarray(B)  # (n_b, n_features)

        # ||xa - xb||^2, shape (n_a, n_b)
        sqdist = np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=-1)

        return self.sigma_f**2 * np.exp(-sqdist / (2 * self.len_scale**2))

    def spectral_density(self, s: NDArray, d: int) -> NDArray:
        """
        Spectral density of the SE kernel (eq. 3-30):
            S_SE(s) = sigma_f^2 * (2*pi*l^2)^(d/2) * exp(-2*pi^2*l^2*s^2)

        Parameters
        ----------
        s : NDArray
            Frequencies at which to evaluate the spectral density.
        d : int
            Dimension of the input x.
        """
        s = np.asarray(s)
        return (
            self.sigma_f**2
            * (2 * np.pi * self.len_scale**2) ** (d / 2)
            * np.exp(-2 * np.pi**2 * self.len_scale**2 * s**2)
        )

In [ ]:
def power_spectral_density(omega: NDArray, ls: float | NDArray, n_dims: int) -> NDArray:
    """
    Power spectral density for the ExpQuad kernel.

    .. math::

        S(\\boldsymbol\\omega) =
            (\\sqrt(2 \\pi)^D \\prod_{i}^{D}\\ell_i
            \\exp\\left( -\\frac{1}{2} \\sum_{i}^{D}\\ell_i^2 \\omega_i^{2} \\right)

    Args:
        omega: array of shape (m_star, d), frequencies at which to evaluate the PSD.
        ls:    lengthscale(s), either a scalar or array of shape (d,).
        n_dims: number of input dimensions D.

    Returns:
        Array of shape (m_star,), one PSD value per basis function.
    """
    ls_arr = np.ones(n_dims) * ls          # (d,)
    c = np.power(np.sqrt(2.0 * np.pi), n_dims)
    exp = np.exp(-0.5 * np.dot(np.square(omega), np.square(ls_arr)))  # (m_star,)
    return c * np.prod(ls_arr) * exp       # (m_star,)

def calc_eigenvalues(L: NDArray, m: Sequence[int]) -> NDArray:
    """Calculate eigenvalues of the Laplacian."""
    temp = [np.arange(1, 1 + m[d]) for d in range(len(m))]
    S = np.meshgrid(*temp)
    S_arr = np.vstack([s.flatten() for s in S]).T
    return np.square((np.pi * S_arr) / (2 * L))


# def calc_eigenvectors(Xs: NDArray, L: NDArray, eigvals: NDArray, m: Sequence[int]) -> NDArray:
#     """Calculate eigenvectors of the Laplacian.

#     These are used as basis vectors in the HSGP approximation.

#     Parameters 
#     --------
#     Xs : NDArray, shape (n_samples, n_features)
#     L : NDArray, shape (n_features, )
#     eigvals
#     """
#     m_star = int(np.prod(m))
#     phi = np.ones((Xs.shape[0], m_star))
#     for d in range(len(m)):
#         c = 1.0 / np.sqrt(L[d])
#         term1 = np.sqrt(eigvals[:, d])
#         term2 = np.tile(Xs[:, d][:, None], m_star) + L[d]
#         phi *= c * np.sin(term1 * term2)
#     return phi

def calc_eigenvectors(Xs: NDArray, L: NDArray, eigvals: NDArray) -> NDArray:
    """Calculate eigenvectors of the Laplacian.
    These are used as basis vectors in the HSGP approximation.

    Parameters
    ----------
    Xs      : NDArray, shape (n_samples, d)
    L       : NDArray, shape (d,)
    eigvals : NDArray, shape (m_star, d)
    m       : Sequence[int], shape (d,)

    Returns
    -------
    phi : NDArray, shape (n_samples, m_star)
    """
    # (1, m_star, d) * (n_samples, 1, d) -> (n_samples, m_star, d)
    term1 = np.sqrt(eigvals)[None, :, :]          # (1, m_star, d)
    term2 = Xs[:, None, :] + L[None, None, :]     # (n_samples, 1, d) + (1, 1, d)
    c = 1.0 / np.sqrt(L)                          # (d,)

    phi = c * np.sin(term1 * term2)               # (n_samples, m_star, d)
    return np.prod(phi, axis=-1)                  # (n_samples, m_star)


In [ ]:
X =np.asarray([
    [0.5, 1.0],
    [-0.3, 0.8]
]) 
m = [3]
L = np.asarray([1.0, 1.5])

eigvals = calc_eigenvalues(L, m)
print(eigvals)
eigvecs = calc_eigenvectors(X, L, eigvals)
print(eigvecs)

In [ ]:
n_dim = 3
bound = 1
m_hat = 60
ls = 1.0
sigma_f = 5
sigma_n = 1e-1
extension = 3
rng = np.random.default_rng(seed=0)

# Define grid points along each axis
n_per_dim = 5

x = np.linspace(-1, 1, n_per_dim)
g = np.meshgrid(*([x]*n_dim))

# Create the full grid and reshape to (n_samples, 3)
X = np.vstack([gi.ravel() for gi in g]).T  # (25, 3)
# X = 2*rng.random((125,3))
X = rng.normal(0, (1,1,.1), size=(125,3))
L = np.ones(n_dim, dtype=float)*extension
m = [m_hat]*n_dim # m_star = m_hat^d = 3375
# m = [25, 25, 25]


In [ ]:

eigvals = calc_eigenvalues(L, m)  # (m_start, d)
print(f"{eigvals.shape = }")


In [ ]:

phi = calc_eigenvectors(X, L, eigvals)  # (n_samples, m_star)
print(phi.shape)
omega = np.sqrt(eigvals)  # (m_star, d)
print(omega.shape)


In [ ]:

psd = power_spectral_density(omega, ls, n_dim)*sigma_f**2
print(psd.shape)


In [ ]:

gram_C = (phi * psd) @ phi.T + np.eye(X.shape[0])*sigma_n**2


In [ ]:
rng = np.random.default_rng(seed=0)

# X = 2*bound*rng.random((100, n_dim)) - bound

gram_B = sigma_f**2*exp_kernel(X, X) + np.eye(X.shape[0])*sigma_n**2


In [ ]:

fig, axs = plt.subplots(1, 3)

axs[0].imshow(gram_B, cmap='viridis', aspect="auto")
axs[0].set_title("Exact Gram Matrix")
axs[1].imshow(gram_C, cmap='viridis', aspect="auto")
axs[1].set_title("Approximate HSGP")
axs[2].imshow(gram_B - gram_C, cmap='viridis', aspect="auto")
axs[2].set_title("Difference")
plt.tight_layout()
plt.show()